# Notebook 3: Temperature acquisition with a thermocouple

**Bassel FADLALLAH**

*Python for Instrumentation and Laboratory Experiments, Pr. T. Angot*

Retrieve the thermocouple voltage every 1 s for at least 900 s, convert it to temperature (type K, ITS-90), plot the data and save them to a file.

No thermocouple was connected to the Keithley during the session, so the acquisition could not be run.

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import serial

## Conversion voltage / temperature

The thermocouple measures relative to the reference junction (the Keithley terminals, at room temperature $T_{ref}$), so:
$$T = E^{-1}\left(V_{measured} + E(T_{ref})\right)$$
with $E(T)$ the type K polynomial (NIST ITS-90) and $E^{-1}$ the inverse polynomial, which has three voltage ranges.

In [2]:
# voltage (mV) -> temperature (°C), coefficients d0..dn
d_neg  = [0, 2.5173462E+1, -1.1662878E+0, -1.0833638E+0, -8.9773540E-1,
          -3.7342377E-1, -8.6632643E-2, -1.0450598E-2, -5.1920577E-4]                   # -5.891 to 0 mV
d_mid  = [0, 2.508355E+1, 7.860106E-2, -2.503131E-1, 8.315270E-2,
          -1.228034E-2, 9.804036E-4, -4.413030E-5, 1.057734E-6, -1.052755E-8]           # 0 to 20.644 mV
d_high = [-1.318058E+2, 4.830222E+1, -1.646031E+0, 5.464731E-2,
          -9.650715E-4, 8.802193E-6, -3.110810E-8]                                        # 20.644 to 54.886 mV

# temperature (°C) -> voltage (mV), 0 to 1372 °C
c_pos = [-1.76004136860E-2, 3.89212049750E-2, 1.85587700320E-5, -9.94575928740E-8,
         3.18409457190E-10, -5.60728448890E-13, 5.60750590590E-16, -3.20207200030E-19,
         9.71511471520E-23, -1.21047212750E-26]
a0, a1, a2 = 1.185976E-1, -1.183432E-4, 1.269686E+2

def temp_to_mV(T):
    return np.polyval(c_pos[::-1], T) + a0*np.exp(a1*(T - a2)**2)

def volt_to_temp(V, T_ref=0.0):
    x = V*1000 + temp_to_mV(T_ref)   # mV
    if x < -5.891 or x > 54.886:
        return np.nan
    if x < 0:
        d = d_neg
    elif x <= 20.644:
        d = d_mid
    else:
        d = d_high
    return float(np.polyval(d[::-1], x))

# check with the reference table: 100 °C = 4.096 mV, 500 °C = 20.644 mV, 1000 °C = 41.276 mV
for T in [100, 500, 1000]:
    print(f"{T} °C -> {temp_to_mV(T):.3f} mV -> {volt_to_temp(temp_to_mV(T)/1000):.2f} °C")

100 °C -> 4.096 mV -> 99.97 °C
500 °C -> 20.644 mV -> 499.95 °C
1000 °C -> 41.276 mV -> 999.98 °C


## Acquisition

In [3]:
def show_live(t_list, T_list):
    clear_output(wait=True)
    plt.plot(t_list, T_list, 'r.-')
    plt.xlabel("time (s)")
    plt.ylabel("temperature (°C)")
    plt.show()

def acquire(ser, duration=900, dt=1.0, T_ref=0.0, live=True):
    ser.write(b":SENS:FUNC 'VOLT:DC'\r")
    ser.write(b":SENS:VOLT:DC:RANG 0.1\r")
    ser.write(b":SENS:VOLT:DC:NPLC 5\r")
    time.sleep(0.5)

    t_list, V_list, T_list = [], [], []
    t0 = time.time()
    k = 0
    try:
        while k*dt <= duration:
            delay = t0 + k*dt - time.time()      # next reading at t0 + k*dt
            if delay > 0:
                time.sleep(delay)
            ser.reset_input_buffer()
            ser.write(b":READ?\r")
            raw = ser.readline().decode('ascii', errors='replace').strip()
            t = time.time() - t0
            k += 1
            try:
                V = float(raw)
            except ValueError:
                print(f"t = {t:6.1f} s: no valid reply, point skipped")
                continue
            T = np.nan if abs(V) > 1e30 else volt_to_temp(V, T_ref)   # +9.9E37 = overflow
            t_list.append(t); V_list.append(V); T_list.append(T)
            if live and len(t_list) % 10 == 0:
                show_live(t_list, T_list)
    except KeyboardInterrupt:
        print("Stopped by hand, data kept.")
    return np.array(t_list), np.array(V_list), np.array(T_list)

def plot_data(t, V, T, title=""):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), sharex=True)
    ax1.plot(t, V*1000, 'b.-', ms=2, lw=0.5)
    ax1.set_ylabel("voltage (mV)")
    ax1.set_title(title)
    ax2.plot(t, T, 'r.-', ms=2, lw=0.5)
    ax2.set_ylabel("temperature (°C)")
    ax2.set_xlabel("time (s)")
    plt.show()

def save_data(filename, t, V, T):
    np.savetxt(filename, np.column_stack((t, V, T)), fmt="%.3f %.8e %.3f",
               header="time_s  voltage_V  temperature_C")
    print(len(t), "points saved to", filename)

Measurement with the real instrument:

In [ ]:
serial_address = "COM3"
T_ref = 22.0   # room temperature, °C

with serial.Serial(serial_address, baudrate=9600, bytesize=8, parity='N', stopbits=1, timeout=2) as ser:
    t, V, T = acquire(ser, duration=900, dt=1.0, T_ref=T_ref)

plot_data(t, V, T, "Cooling of tin")
save_data("tin_thermocouple_data.txt", t, V, T)

Expected curve for tin: the liquid cools down, goes a few degrees below 231.93 °C without solidifying (supercooling), jumps back to 231.93 °C when solidification starts (latent heat), stays on a plateau while liquid and solid coexist, then the solid cools down.